In [1]:
from datasets import load_dataset, concatenate_datasets

data = load_dataset('hlt-lab/voicebench', 'openbookqa')

combined_dataset = data["test"]
combined_dataset[0]

{'audio': {'path': None,
  'array': array([ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00, ...,
          0.00000000e+00, -3.05175781e-05, -3.05175781e-05]),
  'sampling_rate': 16000},
 'prompt': 'There is most likely going to be fog around:\nA. a marsh\nB. a tundra\nC. the plains\nD. a desert\nWhat is the answer to the above multiple choice question? Select one of the following: A, B, C, or D.',
 'reference': 'A'}

In [3]:
import re

def transform_example(example, idx):
    """
    Transforms a single example from the original format to the new format.
    """
    # Extract question and options from the prompt
    prompt_text = example['prompt']
    
    # The question is the first line.
    # The options are the lines starting with A., B., C., D.
    # The last two lines are a generic instruction.
    lines = prompt_text.strip().split('\n')
    
    question = lines[0]
    
    # Find options (e.g., "A. ...", "B. ...")
    options = [line for line in lines if re.match(r'^[A-Z]\.\s', line)]
    
    # Create the prompt for text-to-speech, which includes the question and options
    prompt_for_tts = prompt_text
    
    # Remove the "A. ", "B. " prefixes for the options list
    cleaned_options = [re.sub(r'^[A-Z]\.\s', '', opt) for opt in options]
    
    # Convert the reference character ('A', 'B', etc.) to a zero-based index
    answer_index = ord(example['reference']) - ord('A')

    return {
        'key': "obqa_" + str(idx),
        'question': question,
        'options': cleaned_options,
        'answer_index': answer_index,
        'prompt_for_tts': prompt_for_tts,
        'category': "obqa"
    }

transformed_dataset = combined_dataset.map(transform_example, writer_batch_size=200, with_indices=True, remove_columns=["reference", "prompt"])


Map:   0%|          | 0/455 [00:00<?, ? examples/s]

In [5]:
transformed_dataset[1]

{'audio': {'path': None,
  'array': array([ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00, ...,
          0.00000000e+00,  0.00000000e+00, -3.05175781e-05]),
  'sampling_rate': 16000},
 'key': 'obqa_1',
 'question': 'Predators eat',
 'options': ['lions', 'humans', 'bunnies', 'grass'],
 'answer_index': 2,
 'prompt_for_tts': 'Predators eat\nA. lions\nB. humans\nC. bunnies\nD. grass\nWhat is the answer to the above multiple choice question? Select one of the following: A, B, C, or D.',
 'category': 'obqa'}

In [6]:
transformed_dataset.save_to_disk(os.environ.get("OBQA_DS_PATH", "/path/to/your/dataset/VB_OBQA/full_455_hf_format.v0"))

Saving the dataset (0/1 shards):   0%|          | 0/455 [00:00<?, ? examples/s]